In [ ]:
import torch
from training_functions import train, evaluate_mae_mse
from mcnn import MCNN
import torch.nn as nn
import torch.optim as optim
from dataloader_mcnn import CrowdDataset
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, ConcatDataset
import glob
import os
import numpy as np
import random
import time
import pandas as pd
from pytorch_msssim import ssim
import csv
from rbm import RBM
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error
SEED = 42

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

### Standard training

In [ ]:
main_folders = [r"data\ShanghaiTech_part_A", 
                r"data\ShanghaiTech_part_B"]

train_datasets = []
val_datasets = []

for folder in main_folders:
    train_img_root = os.path.join(folder, "train", "images")
    train_map_root = os.path.join(folder, "train", "maps")
    print(folder)
    if os.path.exists(train_img_root) and os.path.exists(train_map_root):
        print('ok')
        train_datasets.append(CrowdDataset(train_img_root, train_map_root, gt_downsample=4, resize=False))
    
    val_img_root = os.path.join(folder, "test", "images")
    val_map_root = os.path.join(folder, "test", "maps")
    
    if os.path.exists(val_img_root) and os.path.exists(val_map_root):
        val_datasets.append(CrowdDataset(val_img_root, val_map_root, gt_downsample=4, resize=False))

train_dataset = ConcatDataset(train_datasets)
val_dataset = ConcatDataset(val_datasets)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False,pin_memory=True)

In [ ]:
set_seed(SEED)
model_dir = r"output\models\mcnn"
history_dir = r"output\history\mcnn"

os.makedirs(model_dir, exist_ok=True)
os.makedirs(history_dir, exist_ok=True)

model_path = model_dir + r"\mcnn.pth"
history_path = history_dir + r"\mcnn.pkl"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MCNN()
model.eval()
model = model.to(device)
epochs = 100
tolerance = 20

criterion = nn.MSELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


start_time = time.time()
history = train(epochs, model, train_loader, val_loader, optimizer, criterion, device,model_path, tolerance = tolerance)
end_time = time.time()
print(f"total time: {end_time - start_time}\n")

### Boltzman machines

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MCNN()
model.load_state_dict(torch.load(r'output\models\mcnn\mcnn.pth'))
model.eval()
model = model.to(device)

In [ ]:
main_folders = [
    r"data\ShanghaiTech_part_A\train", 
    r"data\ShanghaiTech_part_B\train",
    r"data\NWPU-Crowd\train",
    r"data\UCF-QNRF_ECCV18\train",
    r"data\jhu_crowd_v2.0\train"
]

output_dir = r"output\models\BoltzmanMCNN"
os.makedirs(output_dir, exist_ok=True)

metadata_path = os.path.join(output_dir, "metadata.csv")

model.eval()

with torch.no_grad():
    folder = main_folders[0]

    test_img_root = os.path.join(folder, "images")
    test_map_root = os.path.join(folder, "maps")

    test_dataset = CrowdDataset(
        test_img_root,
        test_map_root,
        gt_downsample=4,
        resize=False,
        one_size=True
    )

    img, gt_dmap = test_dataset[0]
    img = img.unsqueeze(0).to(device)
    pred = model(img)

    prediction_shape = pred.squeeze(0).shape
    prediction_size = pred.squeeze(0).numel()
    del img, pred
    torch.cuda.empty_cache()

total_images = sum(
    len(
        CrowdDataset(
            os.path.join(folder, "images"),
            os.path.join(folder, "maps"),
            gt_downsample=4,
            resize=False,
            one_size=True
        )
    )
    for folder in main_folders
)

prediction_file = os.path.join(output_dir, "predictions.npy")

predictions = np.lib.format.open_memmap(
    prediction_file,
    mode="w+",
    dtype=np.float32,
    shape=(total_images, prediction_size)
)

with open(metadata_path, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "index",
        "dataset",
        "photo",
        "gt_sum"
    ])

    global_idx = 0

    with torch.no_grad():

        for folder in main_folders:

            test_img_root = os.path.join(folder, "images")
            test_map_root = os.path.join(folder, "maps")

            test_dataset = CrowdDataset(
                test_img_root,
                test_map_root,
                gt_downsample=4,
                resize=False,
                one_size=True
            )

            dataset_name = os.path.basename(
                os.path.dirname(folder)
            )

            for idx in range(len(test_dataset)):

                img, gt_dmap = test_dataset[idx]
                img = img.unsqueeze(0).to(device)
                pred = model(img).squeeze(0)
                pred_vector = (
                    pred.detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                    .reshape(-1)
                )

                gt_sum = gt_dmap.sum().item()

                if hasattr(test_dataset, "images"):
                    photo_name = os.path.basename(
                        test_dataset.images[idx]
                    )
                else:
                    photo_name = str(idx)

                predictions[global_idx] = pred_vector

                writer.writerow([
                    global_idx,
                    dataset_name,
                    photo_name,
                    gt_sum
                ])

                global_idx += 1

                del img
                del pred
                del pred_vector
                del gt_dmap

                if global_idx % 100 == 0:
                    predictions.flush()
                    print(f"Processed {global_idx}/{total_images}")

                torch.cuda.empty_cache()

predictions.flush()

del predictions

print("Finished")

In [ ]:
main_folders = [
    r"data\ShanghaiTech_part_A\test", 
    r"data\ShanghaiTech_part_B\test",
    r"data\NWPU-Crowd\val",
    r"data\UCF-QNRF_ECCV18\test",
    r"data\jhu_crowd_v2.0\test",
    r"data\UCF_CC_50"
]

output_dir = r"output\models\BoltzmanMCNN"
os.makedirs(output_dir, exist_ok=True)

metadata_path = os.path.join(output_dir, "metadata_test.csv")

model.eval()

with torch.no_grad():
    folder = main_folders[0]

    test_img_root = os.path.join(folder, "images")
    test_map_root = os.path.join(folder, "maps")

    test_dataset = CrowdDataset(
        test_img_root,
        test_map_root,
        gt_downsample=4,
        resize=False,
        one_size=True
    )

    img, gt_dmap = test_dataset[0]
    img = img.unsqueeze(0).to(device)
    pred = model(img)

    prediction_shape = pred.squeeze(0).shape
    prediction_size = pred.squeeze(0).numel()
    del img, pred
    torch.cuda.empty_cache()

total_images = sum(
    len(
        CrowdDataset(
            os.path.join(folder, "images"),
            os.path.join(folder, "maps"),
            gt_downsample=4,
            resize=False,
            one_size=True
        )
    )
    for folder in main_folders
)

prediction_file = os.path.join(output_dir, "predictions_tests.npy")

predictions = np.lib.format.open_memmap(
    prediction_file,
    mode="w+",
    dtype=np.float32,
    shape=(total_images, prediction_size)
)

with open(metadata_path, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "index",
        "dataset",
        "photo",
        "gt_sum"
    ])

    global_idx = 0

    with torch.no_grad():

        for folder in main_folders:

            test_img_root = os.path.join(folder, "images")
            test_map_root = os.path.join(folder, "maps")

            test_dataset = CrowdDataset(
                test_img_root,
                test_map_root,
                gt_downsample=4,
                resize=False,
                one_size=True
            )

            dataset_name = os.path.basename(
                os.path.dirname(folder)
            )

            for idx in range(len(test_dataset)):

                img, gt_dmap = test_dataset[idx]
                img = img.unsqueeze(0).to(device)
                pred = model(img).squeeze(0)
                pred_vector = (
                    pred.detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                    .reshape(-1)
                )

                gt_sum = gt_dmap.sum().item()

                if hasattr(test_dataset, "images"):
                    photo_name = os.path.basename(
                        test_dataset.images[idx]
                    )
                else:
                    photo_name = str(idx)

                predictions[global_idx] = pred_vector

                writer.writerow([
                    global_idx,
                    dataset_name,
                    photo_name,
                    gt_sum
                ])

                global_idx += 1

                del img
                del pred
                del pred_vector
                del gt_dmap

                if global_idx % 100 == 0:
                    predictions.flush()
                    print(f"Processed {global_idx}/{total_images}")

                torch.cuda.empty_cache()

predictions.flush()

del predictions

print("Finished")

In [ ]:
data = np.load(r'output\models\BoltzmanMCNN\predictions.npy')
tensor_input = torch.from_numpy(data).to(dtype = torch.float64)

In [ ]:
machine1 = RBM(n_visible = tensor_input.size()[1], n_hidden = tensor_input.size()[1]//16, device='cuda')
machine1.fit(tensor_input, 100, 0.00001, verbose = True)
torch.save(machine1, r"output\models\BoltzmanMCNN\machine1.pt")

In [ ]:
tensor_input2 = machine1.draw_hidden(tensor_input)
machine2 = RBM(n_visible = tensor_input2.size()[1], n_hidden = tensor_input2.size()[1]//8, device='cuda')
machine2.fit(tensor_input2, 100, 0.0001, verbose = True)
torch.save(machine2, r"output\models\BoltzmanMCNN\machine2.pt")

In [ ]:
tensor_input3 = machine2.draw_hidden(tensor_input2)
machine3 = RBM(n_visible = tensor_input3.size()[1], n_hidden = tensor_input3.size()[1]//4, device='cuda')
machine3.fit(tensor_input3, 100, 0.001, verbose = True)
torch.save(machine3, r"output\models\BoltzmanMCNN\machine3.pt")

In [ ]:
tensor_final = machine3.draw_hidden(tensor_input3)
torch.save(tensor_final, r"output\models\BoltzmanMCNN\tensor_final_MCNN.pt")

### ML boltzman models

In [ ]:
X = torch.load(r"output\models\BoltzmanMCNN\tensor_final_MCNN.pt").detach().cpu().numpy()
y = pd.read_csv(r'output\models\BoltzmanMCNN\metadata.csv')['gt_sum'].to_numpy()
mask = np.isfinite(y)
X = X[mask]
y = y[mask]

In [ ]:
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mse = mean_squared_error(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MSE": mse
    }


models = {
    "linear_regression": LinearRegression(),

    "svm": SVR(
        kernel="rbf",
        C=1.0,
        epsilon=0.1
    ),

    "decision_tree": DecisionTreeRegressor(
        random_state=42,
        max_depth=None
    ),

    "random_forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

results = []

for name, model in models.items():

    print(f"Training {name}")

    model.fit(X, y)
    y_pred = model.predict(X)
    metrics = calculate_metrics(y, y_pred)
    training_loss = metrics["MSE"]

    model_path = f"output\models\BoltzmanMCNN\{name}.joblib"
    joblib.dump(model, model_path)

    results.append({
        "model": name,
        "training_loss": training_loss,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"]
    })

results_df = pd.DataFrame(results)

print(results_df)